# 03 · 뉴스 소비 건강지수(NCHI) 산출 — 2코어 + 2축 페르소나

> 선행: [`02-variable-mapping.ipynb`](02-variable-mapping.ipynb) · 방법론: [`docs/groundwork/04-research-aggregation-direction.md`](../docs/groundwork/04-research-aggregation-direction.md)(Perplexity Deep Research) · SSOT: [`src/news_health_features.py`](../src/news_health_features.py)

## 확정 설계 (연구 근거)
- **집계 = 기하평균** `NCHI = √(T × D)` (저보완성 — 한 축이 다른 축 결핍을 못 메움; UNDP HDI 2010 전환·JRC-COIN). **산술평균 병기**(강건성, JRC 의무).
- **방향성 = 단조 v1**(높을수록 건강). 신뢰의 역U(맹신·냉소 모두 불건강)와 다양성 상한은 **한계로 기록**(보정모델 v2).
- **2축 페르소나**: 신뢰×다양성 4사분면 — 🟢건강한 소비자 / 🟡신뢰편향형 / 🟠비판적 탐색형 / 🔴이중취약형.
- 정규화 [1,100]·`WT` 가중·연령대 비닝. 선행 한국 사례 없음(최초 시도 가능).

> ⚠️ 분석 수치는 KPF 원자료 재검증 전까지 보고서 직접 인용 금지(내부 설계 근거).

In [1]:
import sys; sys.path.insert(0, "../src"); sys.path.insert(0, "src")
import numpy as np, pandas as pd
import news_health_features as F

df = F.load_2025(); w = df[F.WEIGHT]
band = F.age_band(df)
print(f"로딩 {df.shape[0]:,}행 · 가중치 WT 합={w.sum():,.0f}")

로딩 6,000행 · 가중치 WT 합=6,000


## 1. 신뢰지수 (0~100)

코어 22문항 → z-표준화 → 동일가중 평균 → [1,100]. **동일가중 vs PCA가중이 r=0.999로 동일** → 투명한 동일가중 채택(PCA 불필요).

In [2]:
T = F.trust_index(df); T_pca = F.trust_index(df, weights="pca")
print(f"신뢰지수 가중평균 = {F.wmean(T, w):.1f}")
print(f"  동일가중 vs PCA가중 상관 r = {T.corr(T_pca):.3f}  (≈1 → 동일가중 채택)")
print("연령대별(가중):", {b: round(F.wmean(T[band==b], w[band==b]),1) for b in F.AGE_LABELS})
print("→ 신뢰는 연령대 평탄(62~64)")

신뢰지수 가중평균 = 62.9
  동일가중 vs PCA가중 상관 r = 0.999  (≈1 → 동일가중 채택)
연령대별(가중): {'19-29': 62.7, '30-39': 63.0, '40-49': 62.7, '50-59': 62.7, '60-69': 62.5, '70+': 64.0}
→ 신뢰는 연령대 평탄(62~64)


## 2. 다양성지수 (0~100)

Richness(뉴스 매체유형 폭) → [1,100]. 연령대별 급감(EDA·02 확인).

In [3]:
D = F.diversity_index(df)
print(f"다양성지수 가중평균 = {F.wmean(D, w):.1f}")
print("연령대별(가중):", {b: round(F.wmean(D[band==b], w[band==b]),1) for b in F.AGE_LABELS})
print(f"신뢰 ↔ 다양성 상관 r = {T.corr(D):.3f}  (거의 독립 → 두 축 결합·프로필 의미)")

다양성지수 가중평균 = 27.2
연령대별(가중): {'19-29': 28.0, '30-39': 31.0, '40-49': 31.5, '50-59': 29.6, '60-69': 24.7, '70+': 17.4}
신뢰 ↔ 다양성 상관 r = 0.077  (거의 독립 → 두 축 결합·프로필 의미)


## 3. NCHI 산출 — 기하평균(주) + 산술평균(강건성)

기하평균은 불균형(신뢰高·다양低)을 자동 페널티. 산술평균과 병기해 민감도를 투명 보고.

In [4]:
nchi_g = F.nchi(T, D, "geometric")
nchi_a = F.nchi(T, D, "arithmetic")
print(f"NCHI 기하 가중평균 = {F.wmean(nchi_g, w):.1f}  |  산술 = {F.wmean(nchi_a, w):.1f}  (Δ={F.wmean(nchi_a,w)-F.wmean(nchi_g,w):+.1f})")
print(f"기하~산술 상관 r = {nchi_g.corr(nchi_a):.3f}")
print("\nNCHI(기하) 분포:")
print(nchi_g.describe().round(1).to_string())

NCHI 기하 가중평균 = 39.3  |  산술 = 45.0  (Δ=+5.8)
기하~산술 상관 r = 0.922

NCHI(기하) 분포:
count    6000.0
mean       38.0
std        13.7
min         4.2
25%        26.2
50%        39.5
75%        47.4
max        84.5


In [5]:
# 연령대별 NCHI(가중) — 기하 vs 산술 민감도
rows = []
for b in F.AGE_LABELS:
    m = band == b
    rows.append([b, round(F.wmean(T[m],w[m]),1), round(F.wmean(D[m],w[m]),1),
                 round(F.wmean(nchi_g[m],w[m]),1), round(F.wmean(nchi_a[m],w[m]),1)])
display(pd.DataFrame(rows, columns=["연령대","신뢰","다양성","NCHI기하","NCHI산술"]))
print("→ 40대 정점 → 70+ 최저. 다양성이 NCHI 연령차를 주도(신뢰는 평탄)")

,연령대,신뢰,다양성,NCHI기하,NCHI산술
0,19-29,62.7,28.0,39.5,45.3
1,30-39,63.0,31.0,42.5,47.0
2,40-49,62.7,31.5,42.9,47.1
3,50-59,62.7,29.6,41.3,46.2
4,60-69,62.5,24.7,37.3,43.6
5,70+,64.0,17.4,31.4,40.7


→ 40대 정점 → 70+ 최저. 다양성이 NCHI 연령차를 주도(신뢰는 평탄)


## 4. 2축 페르소나 — 신뢰 × 다양성 4사분면

임계값 = 각 축 **중앙값**(다양성은 Richness 정수라 동값 다수 → 분할 비대칭 가능, 명시).

In [6]:
t_thr, d_thr = T.median(), D.median()
persona = F.persona_quadrant(T, D)
print(f"임계값: 신뢰 중앙={t_thr:.1f}, 다양성 중앙={d_thr:.1f}")
order = ["건강한 소비자","비판적 탐색형","신뢰편향형","이중취약형"]
rows = []
for lab in order:
    share = 100*F.wmean((persona==lab).astype(float), w)
    sub = persona == lab
    rows.append([lab, round(share,1), round(F.wmean(nchi_g[sub],w[sub]),1),
                 round(100*F.wmean(F.avoid_flag(df)[sub].astype(float), w[sub]),2)])
display(pd.DataFrame(rows, columns=["페르소나","가중구성비%","NCHI기하","회피율%"]))

임계값: 신뢰 중앙=63.2, 다양성 중앙=25.8


,페르소나,가중구성비%,NCHI기하,회피율%
0,건강한 소비자,34.8,51.0,0.00
1,비판적 탐색형,28.9,43.5,0.00
2,신뢰편향형,17.8,27.0,6.85
3,이중취약형,18.5,22.3,13.18


In [7]:
# 페르소나 × 연령대 (가중 구성비%)
ct = pd.DataFrame({"persona": persona, "band": band, "w": w}).dropna(subset=["persona","band"])
tab = ct.groupby(["band","persona"], observed=True)["w"].sum().unstack(fill_value=0)
tab = (tab.div(tab.sum(axis=1), axis=0) * 100).round(1)[order]
display(tab)
print("→ 고연령일수록 이중취약형↑·건강한 소비자↓ (다양성 하락 반영)")

persona,건강한 소비자,비판적 탐색형,신뢰편향형,이중취약형
band,,,,
19-29,35.9,29.4,16.6,18.0
30-39,41.8,33.4,9.7,15.1
40-49,41.4,37.1,9.0,12.5
50-59,39.0,32.7,13.0,15.3
60-69,31.9,25.2,20.4,22.5
70+,16.9,14.1,40.1,28.9


→ 고연령일수록 이중취약형↑·건강한 소비자↓ (다양성 하락 반영)


## 5. 방향성 한계 (v1 → v2)

| 축 | v1 처리 | 한계(연구) | v2 보완 |
|----|--------|-----------|---------|
| 신뢰 | 단조(높을수록↑) | 맹신·냉소 모두 불건강, **보정된 신뢰**가 최적(Tsfati & Barnoy 2025) | 회의/리터러시 교차지표로 역U 반영 |
| 다양성 | 단조(높을수록↑) | 과잉(얕은 훑기) 위험 있으나 한국 평균 26.4=하한 위험구간 | 상위 이상치 과잉소비 모니터 |

> v1은 **단조 + 한계 명시**가 방어 가능한 출발점. 깨끗한 '회의' 측정(Q91 보류)이 없어 보정모델은 v2.
> 검증습관·회피는 보조 레이어(02 결정)로 페르소나 태깅에만 사용.

## 6. 결론 · 다음

- **NCHI v1 확정**: 신뢰·다양성 기하평균(가중·연령비닝), 산술 병기. 2축 4사분면 페르소나 가중 구성비 산출.
- 산출물은 SSOT [`src/news_health_features.py`](../src/news_health_features.py)로 재현(노트북은 검증·문서화).

### 다음(04-personas-kmeans)
1. 신뢰·다양성(+보조 검증/회피) 피처로 **K-means 군집** → 4사분면을 데이터 주도 페르소나로 정교화.
2. 페르소나별 인구·매체 프로파일 → 웹 데모(자가진단) 입력.
3. 강건성: 임계값(중앙/평균)·집계(기하/산술)·가중(동일/PCA) 민감도표 정리.